# 2pt N-State Fit Template

This notebook is a runnable example of the existing multi-exponential fit workflow using repository-tracked example correlator data.
Edit the input block below and call the same backend used by the CLI and plain-text input files.


## Imports / Setup


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lqcd_analysis.notebook_workflows import (
    pretty_print_config,
    render_nstate_fit_input_text,
    run_nstate_fit_from_notebook,
    validate_nstate_notebook_config,
)


## User Inputs

The fields below mirror the current plain-text N-state fit input format.
The defaults point to realistic example data included in the repository.


In [2]:
EXAMPLE_DATA = REPO_ROOT / "examples" / "data" / "l64c64a076_m140" / "comb_c2pt_csv"
EXAMPLE_OUTPUTS = REPO_ROOT / "examples" / "outputs" / "nstate_fit_notebook"

workflow_config = {
    # Data settings
    "title_pattern": "l64c64a076_m140_fit_k0_pz*",
    "ns": 64,
    "nt": 64,
    "lattice_spacing_fm": 0.076,
    "c2pt": str(EXAMPLE_DATA / "c2pt_5_5_k0_pz*_real.csv"),
    "pzlist": [0],
    "fold_t": "none",

    # Fit settings
    "model": "normal",
    "fit_mode": "uncorrelated",
    # Recommended default: use a trusted pz=0 E0 and keep fixed-dispersion on.
    "pz0_ground_energy": 0.42,
    "fix_ground_energy_from_dispersion": True,
    "nstates": [1, 2],
    # Recommended default: lock trusted fit windows explicitly per momentum.
    "fit_window": {0: [4, 12]},
    # Advanced / experimental: search plateau inside the retained fit window.
    "auto_search_plateau_from_fit_window": False,
    "binsize": 1,
    "bootstrap_samples": 64,
    "bootstrap_size": 64,
    "seed": 2026,
    "lambda_prior": 1.0,
    "plot": True,

    # Output settings
    "results_dir": str(EXAMPLE_OUTPUTS),
}


## Option Guide

Edit only `workflow_config` in the cell above for normal usage.
The keys are grouped by comments so data settings, fit settings, and output settings stay easy to scan.

- `title_pattern`: Output title pattern. Use `*` where the momentum index `pz` should be inserted.
- `ns`: Spatial lattice extent `Ns`.
- `nt`: Temporal lattice extent `Nt`.
- `lattice_spacing_fm`: Lattice spacing in fm for metadata and summaries.
- `c2pt`: Correlator CSV path or wildcard pattern. Keep `*` in the filename when using multiple `pz` values.
- `pzlist`: List of momentum indices to analyze, for example `[0]` or `[0, 1]`.
- `fold_t`: Time-folding mode before fitting.
  Choices: `"none"` or `False` = no folding; `"periodic"` or `True` = symmetric fold; `"antiperiodic"` = antisymmetric fold.
- `model`: Correlator model.
- Recommended default usage: provide `pz0_ground_energy`, set `fix_ground_energy_from_dispersion` to `True`, and lock trusted windows with `fit_window`. This is the repository’s current legacy-aligned default path.
  Choices: `"normal"` = sum of exponentials; `"symmetric"` = cosh-like forward + backward form; `"antisymmetric"` = sinh-like forward - backward form.
- `fit_mode`: Statistical error model used in the nonlinear fit.
  Choices: `"uncorrelated"` = diagonal fit with per-time-slice bootstrap standard deviations; `"correlated"` = full covariance fit using one shared covariance matrix built from the full bootstrap ensemble and reused for the mean fit and bootstrap fits. If the correlated fit fails for a sample or the window covariance cannot be factorized, the code falls back to a diagonal fit built from the covariance diagonal.
- `pz0_ground_energy`: Optional pz=0 ground-state energy in lattice units. When provided, the 1-state plateau search uses the dispersion target `sqrt(E0_pz0^2 + (2*pi*pz/Ns)^2)` to pre-filter candidate windows before the usual plateau ranking.
- `fix_ground_energy_from_dispersion`: Optional boolean. When `true`, the nonlinear fit also fixes the ground-state energy to that same dispersion target. This is the closest repository-native analogue to the legacy fixed-`E0` setup and is the recommended default when you trust the dispersion anchor.
- `nstates`: Which fits to run. Allowed values are subsets of `[1, 2, 3]`.
  Examples: `[1]`, `[1, 2]`, `[1, 2, 3]`, or `[2]` if you want to rely on cached 1-state results when available.
- `fit_window`: Preferred default notebook-facing fit-window form. Use a dictionary like `{0: [4, 12], 5: [6, 12]}` to set one `[tmin, tmax]` window per momentum. The notebook helper materializes this into the backend fit-window table format automatically.
- `auto_search_plateau_from_fit_window`: Advanced / experimental boolean. When `True`, the workflow first clips the folded correlator to the requested `fit_window`, then runs automatic plateau suggestion inside that retained window. Keep `False` for the current default.
- `binsize`: Integer configuration bin size. Use `1` for no binning.
- `bootstrap_samples`: Number of bootstrap resamples. `None` lets the backend choose automatically.
- `bootstrap_size`: Number of binned configurations drawn per bootstrap sample. `None` uses the backend default.
- `seed`: Random seed for reproducible bootstrap sampling.
- `lambda_prior`: Soft-prior strength for higher-state fits.
  Default: `1.0`. The prior residual is scaled as `sqrt(lambda_prior) * (E - E_prior) / sigma_prior`.
  `1-state` has no lower-state prior, `2-state` gets a prior on `E0`, and `3-state` gets priors on `E0` and `E1` only.
- `plot`: Whether to generate plots automatically.
  Choices: `True` or `False`.
- `results_dir`: Output directory. If omitted or set to `None`, outputs go to the notebook working directory.

Practical note:
- Automatic plateau suggestion remains available through `auto_search_plateau_from_fit_window`, but it should currently be treated as a testing-oriented option rather than the primary recommended default when you already trust the fit window.
- When `fit_window` is provided for a momentum and automatic search is disabled, that `(tmin, tmax)` window is used directly after folding.
- Higher-state fits are initialized hierarchically. If matching lower-state results already exist in the output directory, the code can reuse their plateau information as cache.
- The soft priors affect the actual least-squares objective, not just the initial guesses.
- Fit tables include `fallback_uncorrelated_successes`, the number of bootstrap samples in a given `tmin` window that succeeded only after falling back from the correlated fit to a diagonal fit. The summary reports the same count for the representative window.


## Input Summary / Validation

This notebook follows the same single-config pattern as the TGEVP template.
Fields that belong to the plain-text input file are rendered below; notebook-only runtime fields such as `results_dir` stay in the same config for convenience.


In [3]:
print(pretty_print_config(workflow_config))
print(render_nstate_fit_input_text(workflow_config))
parsed_nstate = validate_nstate_notebook_config(workflow_config)
parsed_nstate


{
  "title_pattern": "l64c64a076_m140_fit_k0_pz*",
  "ns": 64,
  "nt": 64,
  "lattice_spacing_fm": 0.076,
  "c2pt": "/Users/xiang/Desktop/mycodes_CODEX/templates/examples/data/l64c64a076_m140/comb_c2pt_csv/c2pt_5_5_k0_pz*_real.csv",
  "pzlist": [
    0
  ],
  "fold_t": "none",
  "tsrange": [
    0,
    24
  ],
  "model": "normal",
  "fit_mode": "uncorrelated",
  "pz0_ground_energy": null,
  "nstates": [
    1,
    2
  ],
  "tmax": 12,
  "binsize": 1,
  "bootstrap_samples": 64,
  "bootstrap_size": 64,
  "seed": 2026,
  "lambda_prior": 1.0,
  "plot": true,
  "results_dir": "/Users/xiang/Desktop/mycodes_CODEX/templates/examples/outputs/nstate_fit_notebook"
}
l64c64a076_m140_fit_k0_pz* 64 64 0.076
c2pt /Users/xiang/Desktop/mycodes_CODEX/templates/examples/data/l64c64a076_m140/comb_c2pt_csv/c2pt_5_5_k0_pz*_real.csv
pzlist 0
fold_t none
tsrange 0 24
model normal
fit_mode uncorrelated
nstates 1 2
fit_window /path/to/2pt_fit_windows.txt
binsize 1
bootstrap_samples 64
bootstrap_size 64
seed 202

NStateFitInput(title_pattern='l64c64a076_m140_fit_k0_pz*', ns=64, nt=64, lattice_spacing_fm=0.076, correlator_path_pattern='/Users/xiang/Desktop/mycodes_CODEX/templates/examples/data/l64c64a076_m140/comb_c2pt_csv/c2pt_5_5_k0_pz*_real.csv', pzlist=(0,), fold_t='none', tsrange=(0, 24), model='normal', fit_mode='uncorrelated', pz0_ground_energy=None, nstates=(1, 2), tmax=12, binsize=1, bootstrap_samples=64, bootstrap_size=64, seed=2026, lambda_prior=1.0, make_plots=True, results_dir=PosixPath('/Users/xiang/Desktop/mycodes_CODEX/templates/examples/outputs/nstate_fit_notebook'))

## Run Analysis


In [ ]:
nstate_outputs = run_nstate_fit_from_notebook(workflow_config)
for path in nstate_outputs:
    print(path)


## Inspect Outputs

The fit writes tables, bootstrap samples, plots, and a plotting notebook under `examples/outputs/`.


In [ ]:
for path in nstate_outputs:
    print(Path(path).name)
